# noise_reachability step 3: a noise policy trained with SAC (Colab, A100)

Runs `noise_reachability/run_step3_noise_sac.sh` on tasks 7 and 0, one process per task: a small actor on the frozen
`lerobot/smolvla_libero` (`n_action_steps=10`) chooses the initial noise of each executed chunk and is trained with
SAC on task success, 1,500 episodes per task on init states 10-49. Then the final actor and the base policy are
evaluated on init states 0-9 x seeds 0-2, paired. The protocol and hypothesis are in `noise_reachability/README.md`
(step 3); nothing here is a setting to change.

**Before you start.** Runtime > Change runtime type > **A100 GPU** (High-RAM if offered).

**How to run.** Run the cells in order. Step 5 checks the steering on the real model, step 6 runs a short smoke run
(about 10 minutes), step 7 starts the run in the background, and step 8 keeps a cell busy until it ends, so Colab
does not treat the runtime as idle. Keep the tab open and the computer awake. Step 8 prints an estimate of the time
left from the measured episode times.

**If the runtime is lost:** training saves to Google Drive every 25 episodes and evaluation after every episode.
Run steps 1 to 4 and then step 7 again; the run resumes (at most 25 training episodes per task are repeated).
Step 7 refuses to resume on a different commit from the one the run started with.

**Time.** Estimated 20-30 hours for both tasks together (from the step-2 episode times; not measured yet), so the run
will span more than one session. Step 6 and step 8 print the measured time per episode.

## 1. Settings

In [ ]:
import csv, glob, json, os, re, shutil, subprocess, sys, time
from pathlib import Path

REPO_URL = "https://github.com/swagatk/efficient_vla.git"
REPO_REF = "main"
REPO = "/content/efficient_vla"
VENV = "/content/venv"
PY = f"{VENV}/bin/python"

# Everything else (episodes, init states, seeds, SAC settings) comes from the runner and train_noise_sac.py
# defaults, which are the protocol of README.md step 3.
TASKS = "7 0"
DRIVE_DIR = "/content/drive/MyDrive/efficient_vla_runs/noise_sac"
OUT_ROOT = f"{DRIVE_DIR}/run"   # checkpoints, logs and results; the run resumes from them
NUM_EPISODES = 1500             # only for the progress estimate; the runner uses train_noise_sac.py's default

# Commits of the editable installs in the local lerobot_env that produced the existing results.
LEROBOT_COMMIT = "b6ec0060779550c0a157ae34feb89e0cf86012a8"
LIBERO_COMMIT = "8f1084e3132a39270c3a13ebe37270a43ece2a01"
print(f"step 3: tasks {TASKS}, {NUM_EPISODES} training episodes per task, output {OUT_ROOT}")


def sh(cmd, check=True, env=None):
    # Capture and print: notebooks do not show output that a subprocess writes directly.
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, executable="/bin/bash", env=env, capture_output=True, text=True)
    print((r.stdout + r.stderr).rstrip(), flush=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed with exit code {r.returncode}: {cmd}")
    return r

## 2. GPU, RAM and Drive

In [ ]:
smi = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
assert smi.stdout.strip(), "No GPU. Set the runtime type to A100 GPU."
if "A100" not in smi.stdout:
    print("WARNING: not an A100; the run works but is slower")
CUDA_DRIVER = float(re.search(r"CUDA Version: ([0-9.]+)", sh("nvidia-smi").stdout).group(1))
ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
assert ram_gb > 20, f"{ram_gb:.0f} GB RAM; two training processes need about 8 GB each"
print(f"driver supports CUDA {CUDA_DRIVER}; {os.cpu_count()} CPU cores; {ram_gb:.0f} GB RAM")

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(OUT_ROOT, exist_ok=True)

## 3. Code

In [ ]:
if not Path(REPO, ".git").exists():
    sh(f"git clone {REPO_URL} {REPO}")
sh(f"git -C {REPO} fetch origin && git -C {REPO} checkout {REPO_REF} && git -C {REPO} pull --ff-only origin {REPO_REF}")
sh(f"git -C {REPO} log -1 --format='%H %s'")

## 4. Environment

A separate virtual environment with the package versions of the local `lerobot_env` (the same cell as the step-2 notebook). Takes several minutes.

In [ ]:
# Headless rendering: MuJoCo needs the NVIDIA EGL vendor file, which some images lack.
icd = Path("/usr/share/glvnd/egl_vendor.d/10_nvidia.json")
if not icd.exists():
    icd.parent.mkdir(parents=True, exist_ok=True)
    icd.write_text(json.dumps({"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}))
sh("apt-get install -y -qq libegl1 libgles2 libgl1 > /dev/null", check=False)

sh("python -m pip install -q uv")
if not Path(PY).exists():
    sh(f"uv venv {VENV} --python 3.12")

req = Path(REPO, "hybrid_diffusion", "requirements-colab.txt").read_text().splitlines()
if CUDA_DRIVER >= 13.0:
    TORCH_BUILD = "cu130 (as pinned)"
    reqs = req
else:
    cu = "cu128" if CUDA_DRIVER >= 12.8 else "cu126"
    assert CUDA_DRIVER >= 12.6, f"driver supports only CUDA {CUDA_DRIVER}; need 12.6 or newer"
    TORCH_BUILD = f"{cu} (driver too old for cu130)"
    torch_pins = [l for l in req if l.split("==")[0] in ("torch", "torchvision")]
    sh(f"uv pip install -q --python {PY} " + " ".join(f"'{p}+{cu}'" for p in torch_pins)
       + f" --index-url https://download.pytorch.org/whl/{cu} --extra-index-url https://pypi.org/simple")
    reqs = [l for l in req if not re.match(r"(torch|torchvision|triton|nvidia-)", l)]
print("PyTorch build:", TORCH_BUILD)
Path("/tmp/requirements.txt").write_text("\n".join(reqs) + "\n")
sh(f"uv pip install -q --python {PY} --no-deps -r /tmp/requirements.txt")
sh(f"uv pip install -q --python {PY} --no-deps 'lerobot @ git+https://github.com/huggingface/lerobot@{LEROBOT_COMMIT}'")

LIBERO = os.path.expanduser("~/LIBERO")
if not Path(LIBERO, ".git").exists():
    sh(f"git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git {LIBERO}")
sh(f"git -C {LIBERO} checkout -q {LIBERO_COMMIT}")
Path(LIBERO, "libero", "__init__.py").touch()   # present in the local checkout, missing upstream
sh(f"uv pip install -q --python {PY} --no-deps -e {LIBERO}")

# LIBERO asks for these paths interactively on first import unless the config already exists.
root = f"{LIBERO}/libero/libero"
os.makedirs(os.path.expanduser("~/.libero"), exist_ok=True)
os.makedirs(f"{LIBERO}/libero/datasets", exist_ok=True)
Path(os.path.expanduser("~/.libero/config.yaml")).write_text(
    f"benchmark_root: {root}\nbddl_files: {root}/bddl_files\ninit_states: {root}/init_files\n"
    f"datasets: {LIBERO}/libero/datasets\nassets: {root}/assets\n"
)

## 4b. Sanity checks and tests

PyTorch sees the GPU and runs bfloat16, MuJoCo renders headless, and the repo's CPU tests pass (including the step-3 tests).

In [ ]:
RUN_ENV = dict(os.environ, PATH=f"{VENV}/bin:" + os.environ["PATH"], MUJOCO_GL="egl", PYOPENGL_PLATFORM="egl",
               MPLBACKEND="Agg", WANDB_MODE="disabled", PYTHONPATH=REPO, PYTHON_BIN=PY)

sh(f"""{PY} -c "
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))
x = torch.randn(256, 256, device='cuda', dtype=torch.bfloat16)
print('bfloat16 matmul ok', float((x @ x).float().abs().mean()) > 0)
" """, env=RUN_ENV)
egl = sh(f"{PY} -c \"import mujoco; c = mujoco.GLContext(64, 64); c.make_current(); print('EGL context ok')\"", check=False, env=RUN_ENV)
assert "EGL context ok" in egl.stdout, "EGL rendering failed"
sh(f"cd {REPO} && {PY} -m pytest tests/ -q -p no:cacheprovider", env=RUN_ENV)

## 5. Steering check on the real model (about 3 minutes)

`check_step3_steering.py` on one observation of task 7: the wrapper with no chosen noise reproduces the base chunk
bit for bit, a chosen noise is used exactly and repeatably, and the actor's features have the expected size. It
also prints the time per inference with and without steering.

In [ ]:
r = sh(f"cd {REPO} && {PY} -u noise_reachability/check_step3_steering.py", check=False, env=RUN_ENV)
assert r.returncode == 0 and "[check] all passed" in r.stdout, "a steering check failed; see the output above"

## 6. Smoke run (about 10 minutes)

The real runner in a fresh scratch folder: 3 training episodes of task 7 (1 warm-up, then the actor with SAC updates,
small batch), a save and resume across processes, then one evaluation episode each of `base` and `actor`. Either
outcome is fine; not part of the results.

In [ ]:
smoke = "/content/step3_smoke"
shutil.rmtree(smoke, ignore_errors=True)   # a re-run of this cell starts the smoke run again
r = sh(f"bash {REPO}/noise_reachability/run_step3_noise_sac.sh", check=False,
       env=dict(RUN_ENV, SMOKE="1", OUT_ROOT=smoke, EXTRA_ARGS="--episodes_per_process 2"))
log = list(csv.DictReader(open(f"{smoke}/task_7/train_log.csv"))) if Path(smoke, "task_7/train_log.csv").exists() else []
evals = {p: list(csv.DictReader(open(f))) for p in ("base", "actor")
         for f in [f"{smoke}/task_7/eval/per_episode_{p}.csv"] if Path(f).exists()}
ok = (r.returncode == 0 and [int(x["episode"]) for x in log] == [0, 1, 2] and int(log[-1]["updates"]) > 0
      and log[-1]["actor_loss"] != "" and len(evals.get("base", [])) == 1 and len(evals.get("actor", [])) == 1)
assert ok, "smoke run did not finish as expected; see the output above"
print("smoke run ok: training episodes", [(x["success"], x["steps"], x["time_s"]) for x in log],
      "| eval base", evals["base"][0]["success"], "actor", evals["actor"][0]["success"])

## 7. Launch (background)

One runner per task, both at once: training (resumes from the last checkpoint), then the evaluation. Running this
cell while a run is active does nothing.

In [ ]:
running = subprocess.run(["pgrep", "-f", "run_step3_noise_[s]ac"], capture_output=True, text=True).stdout.split()
head = sh(f"git -C {REPO} log -1 --format='%H'").stdout.strip()
first = Path(OUT_ROOT, "repo_commit.txt")
started_with = first.read_text().split()[0] if first.exists() and first.read_text().strip() else None
if running:
    print("A run is already active (pids:", " ".join(running) + "). Not starting another.")
elif started_with and started_with != head:
    # A resumed run must use the code it started with; provenance.jsonl would show the mix, but avoid it.
    raise RuntimeError(f"this run started at commit {started_with}, the checkout is at {head}: in step 3 run "
                       f"`git -C {REPO} checkout {started_with}` (or ask before resuming on new code)")
else:
    log = f"{OUT_ROOT}/runner.log"
    cmd = (f"printf '%s\\n' {TASKS} | xargs -P 2 -I{{}} env TASKS={{}} "
           f"bash {REPO}/noise_reachability/run_step3_noise_sac.sh > /dev/null")
    subprocess.Popen(f"nohup bash -c \"{cmd}; echo runner exit \\$?\" >> '{log}' 2>&1 &", shell=True,
                     executable="/bin/bash", env=dict(RUN_ENV, OUT_ROOT=OUT_ROOT), start_new_session=True)
    sh(f"git -C {REPO} log -1 --format='%H' >> '{OUT_ROOT}/repo_commit.txt'")
    print("Started. Per-task logs:", f"{OUT_ROOT}/task_<t>/train.log and eval.log")

## 8. Progress; waits until the run ends

Every 10 minutes: training episodes done per task, successes in the last 50 (the stochastic actor on training init
states, not a result), the mean episode time and the hours left at that pace, then evaluation episodes. Interrupting
this cell does not stop the run; re-run it any time (needs only steps 1 and 2).

In [ ]:
def progress():
    for t in TASKS.split():
        f = Path(OUT_ROOT, f"task_{t}", "train_log.csv")
        rows = list(csv.DictReader(open(f))) if f.exists() else []
        line = f"task {t}: training {len(rows)}/{NUM_EPISODES}"
        if rows:
            last = rows[-50:]
            secs = sum(float(r["time_s"]) for r in last) / len(last)
            line += (f", last 50: {sum(int(r['success']) for r in last)} ok, {secs:.0f} s/episode, "
                     f"about {(NUM_EPISODES - len(rows)) * secs / 3600:.1f} h of training left")
        for p in ("base", "actor"):
            e = Path(OUT_ROOT, f"task_{t}", "eval", f"per_episode_{p}.csv")
            if e.exists():
                ev = list(csv.DictReader(open(e)))
                line += f" | eval {p} {len(ev)}/30 ({sum(int(r['success']) for r in ev)} ok)"
        print(line, flush=True)

while True:
    print(time.strftime("%H:%M"), flush=True)
    progress()
    if not subprocess.run(["pgrep", "-f", "run_step3_noise_[s]ac"], capture_output=True).stdout:
        break
    time.sleep(600)
print("no runner active")
sh(f"tail -n 3 '{OUT_ROOT}/runner.log'", check=False)

## 9. Results

Standard library only (`python3 -S`): works on a fresh runtime after steps 1 to 3. Fails if an evaluation episode is
missing.

In [ ]:
sh(f"python3 -S {REPO}/noise_reachability/analyze_step3.py --root '{OUT_ROOT}' --tasks {TASKS} "
   f"--step1 {REPO}/week6/noise_reachability_step1_kaggle/step1_reachability.txt.json --out '{OUT_ROOT}/step3_report.txt'")

## 10. Download the results for the repo

Logs, provenance, training logs, evaluation files, the final checkpoints (small) and the step-9 report; the replay
buffers stay on Drive.

In [ ]:
name = "noise_sac_step3_colab"
sh(f"cd '{OUT_ROOT}' && zip -q -r /content/{name}.zip . -x '*.pyc' -x '*replay_*.npz' -x '*.tmp*'")
sh(f"du -h /content/{name}.zip")
from google.colab import files
files.download(f"/content/{name}.zip")